# DeepFlow 순수 예측력 재실험 Colab

이 노트북은 기존 B1/B2/B3 결과의 과대평가 가능성을 제거하기 위한 재실험 파일입니다.

핵심 원칙:

- 모델 단독은 모델만으로 `forecast_demand`를 산출합니다.
- LLM 단독은 LLM만으로 `forecast_demand`를 산출합니다.
- 평가월의 정답 생성 재료는 입력에서 제외합니다.
- 먼저 수요 예측력만 비교합니다.
- 발주량 비교는 예측값을 얻은 뒤 같은 발주 계산식을 적용하는 별도 단계로만 봅니다.

기존 실험에서 제거하는 위험 입력:

- `base_monthly_demand`
- `event_demand_impact_pct`
- `event_lead_time_delta_days`
- `event_capacity_multiplier`
- 재고, 발주, 비용, 기준 판단 관련 컬럼

## 0. 실행 방법

1. 위에서 아래로 실행합니다.
2. B2 셀에서 Gemini API 키를 입력합니다.
3. 이 노트북은 캐시 결과를 사용하지 않습니다.
4. B2는 Gemini API live 호출로만 실행됩니다.
5. Colab 랙을 줄이기 위해 진행상황은 한 줄로만 갱신하고, 그래프는 파일로 저장합니다.

출력 폴더:

- `/content/deepflow_pure_forecast_outputs`

In [ ]:
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "lightgbm", "scikit-learn", "pandas", "matplotlib", "seaborn", "requests"], check=False)
    subprocess.run(["apt-get", "-qq", "update"], check=False)
    subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"], check=False)

In [ ]:
import json
import math
import os
import re
import time
import urllib.request
import zipfile
import base64
from getpass import getpass
from pathlib import Path

import lightgbm as lgb
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from IPython.display import clear_output, display
from sklearn.metrics import mean_absolute_error, mean_squared_error

sns.set_theme(style="whitegrid")

def setup_korean_font():
    candidates = [
        "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
        "/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf",
        "/System/Library/Fonts/AppleSDGothicNeo.ttc",
        "/Library/Fonts/NanumGothic.ttf",
    ]
    for font_path in candidates:
        if Path(font_path).exists():
            fm.fontManager.addfont(font_path)
            font_name = fm.FontProperties(fname=font_path).get_name()
            plt.rcParams["font.family"] = font_name
            plt.rcParams["axes.unicode_minus"] = False
            print(f"한글 폰트 설정 완료: {font_name}")
            return
    plt.rcParams["axes.unicode_minus"] = False
    print("한글 폰트 파일을 찾지 못했습니다.")

setup_korean_font()

TARGET = "synthetic_reference_demand"
FORECAST_HORIZON_MONTHS = 1
TEST_WINDOW_MONTHS = 6
VALIDATION_WINDOW_MONTHS = 6
HISTORY_MONTHS_FOR_LLM = 24
COMPACT_PROGRESS_OUTPUT = True
SHOW_PLOTS_INLINE = False
OUTPUT_DIR = Path("/content/deepflow_pure_forecast_outputs") if IN_COLAB else Path("pure_forecast_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. 데이터 로딩

데이터는 기존 8개 기업 합성데이터를 사용합니다.
단, 이번 재실험에서는 예측 입력을 엄격하게 제한합니다.

In [ ]:
INPUT_PACKAGE_URL = "https://raw.githubusercontent.com/Jaeho777/Impactiveagent/jaeho/scm-ui-rework/public/deepflow_colab_input_package.zip"
EMBEDDED_INPUT_PACKAGE_B64 = """..."""

def download_input_package(data_root: Path) -> None:
    data_root.mkdir(parents=True, exist_ok=True)
    zip_path = data_root / "deepflow_colab_input_package.zip"
    if not zip_path.exists():
        if EMBEDDED_INPUT_PACKAGE_B64:
            print("내장 입력 패키지 사용")
            zip_path.write_bytes(base64.b64decode(EMBEDDED_INPUT_PACKAGE_B64))
        else:
            print(f"입력 패키지 다운로드: {INPUT_PACKAGE_URL}")
            urllib.request.urlretrieve(INPUT_PACKAGE_URL, zip_path)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(data_root)

DATA_ROOT = Path("/content/deepflow_colab_data") if IN_COLAB else Path("colab/deepflow_colab_data")
local_dataset_dir = Path("outputs/final_8_company_datasets_2026-05-23/datasets")

if local_dataset_dir.exists():
    DATASET_DIR = local_dataset_dir
    print(f"로컬 데이터셋 사용: {DATASET_DIR}")
else:
    if not list(DATA_ROOT.rglob("datasets/*.csv")):
        download_input_package(DATA_ROOT)
    candidates = [p for p in DATA_ROOT.rglob("datasets") if list(p.glob("*.csv"))]
    if not candidates:
        raise FileNotFoundError("datasets/*.csv를 찾지 못했습니다.")
    DATASET_DIR = candidates[0]
    print(f"다운로드 데이터셋 사용: {DATASET_DIR}")

csv_files = sorted(DATASET_DIR.glob("*.csv"))
print("CSV 파일 수:", len(csv_files))
for path in csv_files:
    print("-", path.name)

In [ ]:
def read_datasets(dataset_dir: Path) -> pd.DataFrame:
    frames = []
    for path in sorted(dataset_dir.glob("*.csv")):
        frame = pd.read_csv(path)
        frame["dataset_file"] = path.name
        frames.append(frame)
    if not frames:
        raise RuntimeError(f"No CSV files found in {dataset_dir}")
    df = pd.concat(frames, ignore_index=True)
    df["month_dt"] = pd.to_datetime(df["month"] + "-01")
    df = df.sort_values(["company_id", "sku_family", "month_dt"]).reset_index(drop=True)
    return df

df_raw = read_datasets(DATASET_DIR)
print("전체 행 수:", len(df_raw))
print("기업 수:", df_raw["company_id"].nunique())
print("기업-SKU 수:", df_raw[["company_id", "sku_family"]].drop_duplicates().shape[0])
print("월 범위:", df_raw["month"].min(), "~", df_raw["month"].max())
display(df_raw.head(3))

## 2. 순수 예측 feature 생성

이번 실험에서 모델과 LLM은 평가월의 정답 생성 재료를 쓰지 않습니다.

허용:

- 과거 월간 출고량
- 과거 사건 정보
- 기업, 산업, SKU 식별 정보
- 월/시간 정보

금지:

- 평가월 `base_monthly_demand`
- 평가월 `event_demand_impact_pct`
- 평가월 재고, 발주, 비용, 기준 판단 컬럼

In [ ]:
FORBIDDEN_FORECAST_INPUTS = {
    "base_monthly_demand",
    "event_demand_impact_pct",
    "event_lead_time_delta_days",
    "event_capacity_multiplier",
    "opening_inventory",
    "inbound_qty",
    "available_inventory",
    "effective_lead_time_days",
    "moq",
    "lot_multiple",
    "capacity_qty",
    "holding_cost_per_unit",
    "stockout_cost_per_unit",
    "safety_stock",
    "target_inventory_position",
    "reference_order_qty",
    "projected_ending_inventory",
    "projected_shortage_qty",
    "reference_cost",
}

MODEL_CATEGORICAL_FEATURES = [
    "company_id",
    "industry_id",
    "industry_segment",
    "company_archetype",
    "sku_family",
]

MODEL_NUMERIC_FEATURES = [
    "lag_1",
    "lag_2",
    "lag_3",
    "lag_6",
    "rolling_mean_3",
    "rolling_mean_6",
    "rolling_std_3",
    "rolling_std_6",
    "past_event_demand_impact_1",
    "past_event_lead_time_delta_1",
    "past_event_capacity_multiplier_1",
    "month_sin",
    "month_cos",
    "time_index",
]

def add_pure_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["month_number"] = df["month_dt"].dt.month
    df["month_sin"] = np.sin(2 * np.pi * df["month_number"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month_number"] / 12)
    df["time_index"] = df.groupby(["company_id", "sku_family"]).cumcount()
    grouped = df.groupby(["company_id", "sku_family"], group_keys=False)
    df["lag_1"] = grouped[TARGET].shift(1)
    df["lag_2"] = grouped[TARGET].shift(2)
    df["lag_3"] = grouped[TARGET].shift(3)
    df["lag_6"] = grouped[TARGET].shift(6)
    df["rolling_mean_3"] = grouped[TARGET].transform(lambda s: s.shift(1).rolling(3).mean())
    df["rolling_mean_6"] = grouped[TARGET].transform(lambda s: s.shift(1).rolling(6).mean())
    df["rolling_std_3"] = grouped[TARGET].transform(lambda s: s.shift(1).rolling(3).std())
    df["rolling_std_6"] = grouped[TARGET].transform(lambda s: s.shift(1).rolling(6).std())
    df["past_event_demand_impact_1"] = grouped["event_demand_impact_pct"].shift(1)
    df["past_event_lead_time_delta_1"] = grouped["event_lead_time_delta_days"].shift(1)
    df["past_event_capacity_multiplier_1"] = grouped["event_capacity_multiplier"].shift(1)
    fill_base = grouped[TARGET].transform(lambda s: s.shift(1)).fillna(df[TARGET].median())
    for col in ["lag_1", "lag_2", "lag_3", "lag_6", "rolling_mean_3", "rolling_mean_6"]:
        df[col] = df[col].fillna(fill_base)
    for col in ["rolling_std_3", "rolling_std_6", "past_event_demand_impact_1", "past_event_lead_time_delta_1", "past_event_capacity_multiplier_1"]:
        df[col] = df[col].fillna(0)
    return df

def assign_split(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["split"] = "train"
    for _, index in df.groupby(["company_id", "sku_family"]).groups.items():
        ordered = list(index)
        if len(ordered) < 18:
            raise RuntimeError("각 기업-SKU 시계열은 최소 18개월 이상이어야 합니다.")
        validation_index = ordered[-(VALIDATION_WINDOW_MONTHS + TEST_WINDOW_MONTHS):-TEST_WINDOW_MONTHS]
        test_index = ordered[-TEST_WINDOW_MONTHS:]
        df.loc[validation_index, "split"] = "validation"
        df.loc[test_index, "split"] = "test"
    return df

df = assign_split(add_pure_time_features(df_raw))
display(df.groupby("split").size().reset_index(name="rows"))

## 3. 누수 방지 감사

이 셀은 금지 컬럼이 모델 feature에 들어가지 않았는지 확인합니다.
또한 기존 실험에서 문제가 된 oracle형 산식이 얼마나 쉬운 문제였는지도 같이 보여줍니다.

In [ ]:
def mape(actual, pred):
    actual = np.asarray(actual, dtype=float)
    pred = np.asarray(pred, dtype=float)
    return float(np.mean(np.abs(actual - pred) / np.maximum(actual, 1)))

def rmse(actual, pred):
    return float(np.sqrt(mean_squared_error(actual, pred)))

def audit_pure_features(df: pd.DataFrame) -> pd.DataFrame:
    feature_set = set(MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES)
    rows = []
    rows.append({
        "check": "target_not_in_model_features",
        "passed": TARGET not in feature_set,
        "detail": TARGET,
    })
    overlap = sorted(feature_set.intersection(FORBIDDEN_FORECAST_INPUTS))
    rows.append({
        "check": "forbidden_current_month_inputs_removed",
        "passed": len(overlap) == 0,
        "detail": ", ".join(overlap) if overlap else "none",
    })
    ordered = df.sort_values(["company_id", "sku_family", "month_dt"]).copy()
    split_order_ok = True
    for _, group in ordered.groupby(["company_id", "sku_family"]):
        train_max = group[group["split"].isin(["train", "validation"])]["month_dt"].max()
        test_min = group[group["split"] == "test"]["month_dt"].min()
        split_order_ok = split_order_ok and bool(train_max < test_min)
    rows.append({
        "check": "train_validation_before_test",
        "passed": split_order_ok,
        "detail": "all train/validation months are earlier than test months",
    })
    return pd.DataFrame(rows)

audit_df = audit_pure_features(df)
display(audit_df)
assert audit_df["passed"].all()

test_df = df[df["split"] == "test"].copy()
oracle_base = test_df["base_monthly_demand"]
oracle_base_event = test_df["base_monthly_demand"] * (1 + test_df["event_demand_impact_pct"])
oracle_check = pd.DataFrame([
    {"not_used_baseline": "base_monthly_demand_only", "mape": mape(test_df[TARGET], oracle_base)},
    {"not_used_baseline": "base_monthly_demand_x_event", "mape": mape(test_df[TARGET], oracle_base_event)},
])
display(oracle_check)
print("주의: 위 oracle형 baseline은 이번 순수 실험에서 사용하지 않습니다.")

## 4. B1: 직전 3개월 평균

B1은 단순 기준선입니다. 직전 3개월 출고량 평균만 사용합니다.

In [ ]:
def build_prediction_rows(test_df: pd.DataFrame, method: str, forecasts: dict[int, float], raw_source: dict[int, str] | None = None) -> pd.DataFrame:
    rows = []
    for idx, row in test_df.iterrows():
        actual = float(row[TARGET])
        forecast = max(0, float(forecasts[idx]))
        rows.append({
            "row_index": idx,
            "method": method,
            "company_id": row["company_id"],
            "company_name": row["company_name"],
            "industry_id": row["industry_id"],
            "sku_family": row["sku_family"],
            "month": row["month"],
            "month_dt": row["month_dt"],
            "actual_demand": round(actual, 4),
            "forecast_demand": round(forecast, 4),
            "forecast_ape": round(abs(actual - forecast) / actual if actual else 0, 6),
            "raw_source": "" if raw_source is None else raw_source.get(idx, ""),
        })
    return pd.DataFrame(rows)

def run_b1(df: pd.DataFrame) -> pd.DataFrame:
    test_df = df[df["split"] == "test"].copy()
    forecasts = {idx: float(row["rolling_mean_3"]) for idx, row in test_df.iterrows()}
    return build_prediction_rows(test_df, "b1_three_month_avg", forecasts)

b1_rows = run_b1(df)
print("B1 rows:", len(b1_rows))
display(b1_rows.head(3))

## 5. B3: 모델 단독 예측

B3는 LightGBM 모델만으로 `forecast_demand`를 산출합니다.

금지 컬럼은 사용하지 않습니다.
발주 계산도 이 단계에서는 수행하지 않습니다.

In [ ]:
def run_b3_pure_model(df: pd.DataFrame) -> tuple[pd.DataFrame, lgb.LGBMRegressor]:
    train_df = df[df["split"].isin(["train", "validation"])].copy()
    test_df = df[df["split"] == "test"].copy()
    features = MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES
    x_train = pd.get_dummies(train_df[features], columns=MODEL_CATEGORICAL_FEATURES)
    x_test = pd.get_dummies(test_df[features], columns=MODEL_CATEGORICAL_FEATURES)
    x_test = x_test.reindex(columns=x_train.columns, fill_value=0)
    model = lgb.LGBMRegressor(
        objective="regression",
        n_estimators=240,
        learning_rate=0.045,
        num_leaves=31,
        min_child_samples=8,
        random_state=42,
        verbose=-1,
    )
    model.fit(x_train, train_df[TARGET])
    pred = model.predict(x_test)
    forecasts = dict(zip(test_df.index.tolist(), pred))
    output = build_prediction_rows(test_df, "b3_lgbm_pure_model", forecasts)
    importance = pd.Series(model.feature_importances_, index=x_train.columns).sort_values(ascending=False).reset_index()
    importance.columns = ["feature", "importance"]
    return output, model, importance

b3_rows, b3_model, b3_importance = run_b3_pure_model(df)
print("B3 rows:", len(b3_rows))
display(b3_rows.head(3))
display(b3_importance.head(10))

## 6. B2: LLM 단독 예측

B2는 Gemini API live 호출로만 실행합니다.

LLM 프롬프트에 들어가는 것:

- 회사와 산업 정보
- SKU 이름
- 과거 24개월 출고량
- 과거 사건 정보
- 예측 대상 월

LLM 프롬프트에 들어가지 않는 것:

- 평가월 `base_monthly_demand`
- 평가월 사건 영향률
- 평가월 재고, 발주, 비용, 기준 판단
- 모델 예측값
- 계산 도구 결과

In [ ]:
RUN_LLM_CALLS = True
GEMINI_MODEL = "gemini-flash-latest"
REQUEST_DELAY_SECONDS = 10
MAX_GEMINI_RETRIES = 4
RETRY_BASE_SECONDS = 20
RUN_GEMINI_SMOKE_TEST = False

if RUN_LLM_CALLS:
    if not os.environ.get("GEMINI_API_KEY"):
        os.environ["GEMINI_API_KEY"] = getpass("Gemini API Key 입력: ")

B2_RAW_DIR = OUTPUT_DIR / "b2_raw_llm_responses"
B2_RAW_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def extract_json(text: str):
    cleaned = text.strip()
    cleaned = re.sub(r"^```(?:json)?", "", cleaned).strip()
    cleaned = re.sub(r"```$", "", cleaned).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        match = re.search(r"(\[[\s\S]*\]|\{[\s\S]*\})", cleaned)
        if not match:
            raise
        return json.loads(match.group(1))

def flatten_items(parsed) -> list[dict]:
    if isinstance(parsed, list):
        return [item for item in parsed if isinstance(item, dict)]
    if isinstance(parsed, dict):
        for key in ["results", "items", "predictions", "forecasts", "data"]:
            if isinstance(parsed.get(key), list):
                return [item for item in parsed[key] if isinstance(item, dict)]
        return [parsed]
    return []

def prompt_for_b2_company_month(history: pd.DataFrame, test_rows: pd.DataFrame, forecast_month: str) -> str:
    history_by_sku = {}
    for sku_family, sku_history in history.sort_values("month_dt").groupby("sku_family"):
        history_by_sku[sku_family] = [
            {
                "month": row.month,
                "monthly_shipment_qty": int(row.synthetic_reference_demand),
                "past_event_type": row.event_type,
                "past_event_demand_impact_pct": float(row.event_demand_impact_pct),
            }
            for row in sku_history.tail(HISTORY_MONTHS_FOR_LLM).itertuples()
        ]
    meta = test_rows.iloc[0]
    target_rows = [
        {
            "target_no": position + 1,
            "row_index": int(index),
            "sku_family": row["sku_family"],
            "month": row["month"],
        }
        for position, (index, row) in enumerate(test_rows.sort_values("sku_family").iterrows())
    ]
    payload = {
        "task": "Pure one-step demand forecasting. Predict only forecast_demand for the given forecast_month. Do not calculate orders. Return one strict JSON array only.",
        "company_id": meta["company_id"],
        "company_name": meta["company_name"],
        "industry_id": meta["industry_id"],
        "forecast_horizon_months": FORECAST_HORIZON_MONTHS,
        "forecast_month": forecast_month,
        "rules": [
            "Use only company context, sku names, target month, and history_last_24_months_by_sku.",
            "Do not ask for external tools.",
            "Do not output recommended_order_qty.",
            "forecast_demand must be an integer.",
            "Do not use thousands separators.",
            "Return valid JSON only. Do not include markdown fences, comments, or prose.",
        ],
        "history_last_24_months_by_sku": history_by_sku,
        "target_rows": target_rows,
        "output_schema": [
            {
                "target_no": "integer",
                "row_index": "integer",
                "sku_family": "string",
                "month": "YYYY-MM",
                "forecast_demand": "integer",
            }
        ],
    }
    forbidden_target_keys = FORBIDDEN_FORECAST_INPUTS.union({"recommended_order_qty", "reference_order_qty"})
    target_key_overlap = sorted(set().union(*(set(item.keys()) for item in target_rows)).intersection(forbidden_target_keys))
    if target_key_overlap:
        raise RuntimeError(f"B2 target_rows contains forbidden keys: {target_key_overlap}")
    return json.dumps(payload, ensure_ascii=False)

def call_gemini_json_once(prompt: str, model: str = GEMINI_MODEL) -> str:
    api_key = os.environ.get("GEMINI_API_KEY", "")
    if not api_key:
        raise RuntimeError("GEMINI_API_KEY가 설정되지 않았습니다.")
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent"
    body = {
        "contents": [
            {
                "role": "user",
                "parts": [{"text": "Return one strict JSON array only.\n\n" + prompt}],
            }
        ],
        "generationConfig": {
            "temperature": 0,
            "responseMimeType": "application/json",
        },
    }
    response = requests.post(url, params={"key": api_key}, json=body, timeout=80)
    if response.status_code >= 400:
        raise RuntimeError(f"Gemini API error {response.status_code}: {response.text[:500]}")
    payload = response.json()
    return payload["candidates"][0]["content"]["parts"][0]["text"]

def call_gemini_json(prompt: str, model: str = GEMINI_MODEL) -> str:
    last_error = None
    for attempt in range(1, MAX_GEMINI_RETRIES + 1):
        try:
            return call_gemini_json_once(prompt, model)
        except RuntimeError as error:
            message = str(error)
            last_error = error
            if "Gemini API error 403" in message:
                raise RuntimeError("Gemini API 403: 키 또는 권한 문제입니다.") from error
            if "Gemini API error 429" not in message:
                raise
            wait_seconds = RETRY_BASE_SECONDS * attempt
            print(f"Gemini quota 대기: {wait_seconds}초 ({attempt}/{MAX_GEMINI_RETRIES})")
            time.sleep(wait_seconds)
    raise last_error

In [ ]:
def run_b2_pure_llm(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    if not RUN_LLM_CALLS:
        raise RuntimeError("RUN_LLM_CALLS=False입니다. 이 노트북은 live LLM 호출 기준으로 실행해야 합니다.")
    test_df = df[df["split"] == "test"].copy()
    forecasts: dict[int, float] = {}
    raw_source: dict[int, str] = {}
    logs = []
    tasks = []
    for (company_id, forecast_month), group in test_df.groupby(["company_id", "month"], sort=True):
        history = df[
            (df["company_id"] == company_id)
            & (df["month_dt"] < group["month_dt"].min())
        ].copy()
        tasks.append((company_id, forecast_month, history, group.copy()))

    print("B2 live 호출 수:", len(tasks))
    for i, (company_id, forecast_month, history, group) in enumerate(tasks, start=1):
        prompt = prompt_for_b2_company_month(history, group, forecast_month)
        try:
            text = call_gemini_json(prompt)
            (B2_RAW_DIR / f"{company_id}_{forecast_month}.json.txt").write_text(text, encoding="utf-8")
            parsed = extract_json(text)
            items = flatten_items(parsed)
            by_row = {}
            for item in items:
                row_index = int(item.get("row_index"))
                by_row[row_index] = item
            missing = 0
            for idx in group.index:
                item = by_row.get(int(idx))
                if item is None:
                    missing += 1
                    continue
                forecasts[idx] = float(item["forecast_demand"])
                raw_source[idx] = f"{company_id}_{forecast_month}.json.txt"
            status = "ok" if missing == 0 else "partial"
            logs.append({"company_id": company_id, "forecast_month": forecast_month, "status": status, "message": f"added={len(group)-missing}, missing={missing}"})
            if COMPACT_PROGRESS_OUTPUT:
                clear_output(wait=True)
                print(f"B2 Gemini live 호출 진행: {i}/{len(tasks)} | {company_id} {forecast_month} | {status}")
            else:
                print(f"[{i}/{len(tasks)}] {company_id} {forecast_month}: {status}")
        except Exception as error:
            logs.append({"company_id": company_id, "forecast_month": forecast_month, "status": "fail", "message": str(error)})
            if COMPACT_PROGRESS_OUTPUT:
                clear_output(wait=True)
                print(f"B2 Gemini live 호출 실패: {i}/{len(tasks)} | {company_id} {forecast_month}")
                print(error)
            else:
                print(f"[{i}/{len(tasks)}] {company_id} {forecast_month}: fail {error}")
            raise
        time.sleep(REQUEST_DELAY_SECONDS)

    missing_index = sorted(set(test_df.index.tolist()) - set(forecasts.keys()))
    if missing_index:
        raise RuntimeError(f"B2 missing forecasts: {missing_index[:10]}")
    return build_prediction_rows(test_df, "b2_gemini_pure_llm", forecasts, raw_source), pd.DataFrame(logs)

# Gemini smoke test
if RUN_LLM_CALLS and RUN_GEMINI_SMOKE_TEST:
    smoke = call_gemini_json_once('[{"target_no":1,"row_index":1,"sku_family":"test","month":"2026-01","forecast_demand":100}]')
    print("Gemini smoke test OK")

b2_rows, b2_call_log = run_b2_pure_llm(df)
display(b2_call_log.tail(10))
display(b2_rows.head(3))

## 7. 순수 예측력 지표 계산

이 표는 수요 예측값만 비교합니다.
발주량은 여기서 비교하지 않습니다.

In [ ]:
def summarize_forecast_metrics(combined: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for method, group in combined.groupby("method"):
        rows.append({
            "method": method,
            "test_rows": len(group),
            "forecast_mape": round(float(group["forecast_ape"].mean()), 6),
            "forecast_mae": round(float(mean_absolute_error(group["actual_demand"], group["forecast_demand"])), 4),
            "forecast_rmse": round(rmse(group["actual_demand"], group["forecast_demand"]), 4),
        })
    order = {"b1_three_month_avg": 1, "b2_gemini_pure_llm": 2, "b3_lgbm_pure_model": 3}
    metrics = pd.DataFrame(rows)
    metrics["sort_order"] = metrics["method"].map(order)
    return metrics.sort_values("sort_order").drop(columns=["sort_order"])

combined = pd.concat([b1_rows, b2_rows, b3_rows], ignore_index=True)
metrics = summarize_forecast_metrics(combined)
display(metrics)

b1_rows.to_csv(OUTPUT_DIR / "b1_three_month_avg_pure_predictions.csv", index=False, encoding="utf-8-sig")
b2_rows.to_csv(OUTPUT_DIR / "b2_gemini_pure_llm_predictions.csv", index=False, encoding="utf-8-sig")
b3_rows.to_csv(OUTPUT_DIR / "b3_lgbm_pure_model_predictions.csv", index=False, encoding="utf-8-sig")
b2_call_log.to_csv(OUTPUT_DIR / "b2_gemini_pure_llm_call_log.csv", index=False, encoding="utf-8-sig")
combined.to_csv(OUTPUT_DIR / "combined_pure_forecast_predictions.csv", index=False, encoding="utf-8-sig")
metrics.to_csv(OUTPUT_DIR / "metrics_pure_forecast.csv", index=False, encoding="utf-8-sig")
print("저장 위치:", OUTPUT_DIR)

## 8. 선택 사항: 같은 발주 계산식 적용 후 발주 지표 확인

이 단계는 순수 예측력 비교가 아닙니다.
예측값을 얻은 뒤, 동일한 발주 계산식을 적용했을 때 운영 지표가 어떻게 바뀌는지 보는 별도 분석입니다.

In [ ]:
def ceil_to_multiple(value: float, multiple: float) -> int:
    multiple = max(1, int(round(multiple)))
    if value <= 0:
        return 0
    return int(math.ceil(value / multiple) * multiple)

def floor_to_multiple(value: float, multiple: float) -> int:
    multiple = max(1, int(round(multiple)))
    if value <= 0:
        return 0
    return int(math.floor(value / multiple) * multiple)

def fair_recommend_order(row: pd.Series, forecast_demand: float) -> int:
    lead_time = float(row["effective_lead_time_days"])
    moq = float(row["moq"])
    lot = float(row["lot_multiple"])
    capacity = float(row["capacity_qty"])
    available = float(row["available_inventory"])
    safety_stock = max(0, round(forecast_demand * (0.16 + min(lead_time, 35) / 170)))
    lead_time_demand = round(forecast_demand * lead_time / 30)
    target_position = max(safety_stock, lead_time_demand + safety_stock)
    raw_order = max(0, target_position - available)
    if raw_order <= 0:
        return 0
    order_qty = ceil_to_multiple(max(raw_order, moq), lot)
    return min(order_qty, floor_to_multiple(capacity, lot))

test_lookup = df[df["split"] == "test"].copy()
enriched = combined.merge(
    test_lookup[["company_id", "sku_family", "month", "reference_order_qty", "available_inventory", "effective_lead_time_days", "moq", "lot_multiple", "capacity_qty"]],
    on=["company_id", "sku_family", "month"],
    how="left",
)
recommended = []
violation = []
for row in enriched.itertuples(index=False):
    order_qty = fair_recommend_order(pd.Series(row._asdict()), row.forecast_demand)
    recommended.append(order_qty)
    bad = order_qty > 0 and (order_qty < row.moq or order_qty % row.lot_multiple != 0 or order_qty > row.capacity_qty)
    violation.append(bad)
enriched["fair_recommended_order_qty"] = recommended
enriched["fair_constraint_violation"] = violation
enriched["fair_order_error_pct"] = enriched.apply(
    lambda r: 0 if r["reference_order_qty"] == 0 and r["fair_recommended_order_qty"] == 0
    else (1 if r["reference_order_qty"] == 0 else abs(r["reference_order_qty"] - r["fair_recommended_order_qty"]) / abs(r["reference_order_qty"])),
    axis=1,
)
fair_order_metrics = enriched.groupby("method", as_index=False).agg(
    fair_order_error_pct=("fair_order_error_pct", "mean"),
    fair_constraint_violation_rate=("fair_constraint_violation", "mean"),
)
display(fair_order_metrics)
enriched.to_csv(OUTPUT_DIR / "pure_forecast_with_common_order_calculation.csv", index=False, encoding="utf-8-sig")
fair_order_metrics.to_csv(OUTPUT_DIR / "fair_order_metrics_after_pure_forecast.csv", index=False, encoding="utf-8-sig")

## 9. 시계열 비교 시각화

대표 5개 기업-SKU에 대해 실제값과 B1/B2/B3 순수 예측값을 비교합니다.

In [ ]:
FIG_DIR = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

def select_representative_keys(combined: pd.DataFrame, top_n: int = 5) -> pd.DataFrame:
    b1 = combined[combined["method"] == "b1_three_month_avg"].copy()
    rank = (
        b1.groupby(["company_id", "company_name", "sku_family"], as_index=False)
        .agg(mean_b1_mape=("forecast_ape", "mean"), mean_actual=("actual_demand", "mean"))
        .sort_values(["mean_b1_mape", "mean_actual"], ascending=[False, False])
        .head(top_n)
    )
    return rank

selected = select_representative_keys(combined, 5)
display(selected)

method_labels = {
    "b1_three_month_avg": "B1 3개월 평균",
    "b2_gemini_pure_llm": "B2 LLM 단독",
    "b3_lgbm_pure_model": "B3 모델 단독",
}
method_colors = {
    "b1_three_month_avg": "#6b7280",
    "b2_gemini_pure_llm": "#dc2626",
    "b3_lgbm_pure_model": "#2563eb",
}

figure_paths = []
for i, row in enumerate(selected.itertuples(index=False), start=1):
    company_id = row.company_id
    sku_family = row.sku_family
    company_name = row.company_name
    actual_series = df[(df["company_id"] == company_id) & (df["sku_family"] == sku_family)].sort_values("month_dt").tail(24)
    pred_series = combined[(combined["company_id"] == company_id) & (combined["sku_family"] == sku_family)].copy()
    pred_series["month_dt"] = pd.to_datetime(pred_series["month"] + "-01")

    plt.figure(figsize=(13, 5.5))
    plt.plot(actual_series["month_dt"], actual_series[TARGET], color="#111827", linewidth=2.4, marker="o", label="실제 월간 출고량")
    test_start = pred_series["month_dt"].min()
    plt.axvspan(test_start, pred_series["month_dt"].max(), color="#fef3c7", alpha=0.35, label="평가 구간")
    for method, label in method_labels.items():
        one = pred_series[pred_series["method"] == method].sort_values("month_dt")
        plt.plot(one["month_dt"], one["forecast_demand"], linestyle="--", linewidth=2, marker="s", color=method_colors[method], label=label)
    plt.title(f"순수 예측 비교 {i}: {company_name} / {sku_family} (horizon=1개월, 평가점=6개)", fontsize=15, pad=14)
    plt.xlabel("월")
    plt.ylabel("월간 출고량")
    plt.legend(loc="best")
    plt.tight_layout()
    path = FIG_DIR / f"pure_forecast_compare_{i}_{company_id}_{sku_family}.png"
    plt.savefig(path, dpi=180, bbox_inches="tight")
    if SHOW_PLOTS_INLINE:
        plt.show()
    else:
        plt.close()
    figure_paths.append(path)

for path in figure_paths:
    print(path)

## 10. 출력 압축

In [ ]:
import shutil

zip_path = shutil.make_archive(str(OUTPUT_DIR), "zip", OUTPUT_DIR)
print("ZIP:", zip_path)

if IN_COLAB:
    from google.colab import files
    files.download(zip_path)